In [10]:
#| default_exp elastic

In [11]:
#| hide

import numpy as np
import igl
from pathlib import Path

In [12]:
#| export

import jax
import jax.numpy as jnp

In [13]:
#| hide

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", False)
jax.config.update('jax_log_compiles', False) # use this to log JAX JIT compilations.


In [14]:
#| export

from jaxtyping import Bool, Float, Int

In [15]:
#| export

from triangulax import trigonometry as trig
from triangulax import geometry as geom
from triangulax.triangular import TriMesh
from triangulax.mesh import HeMesh, GeomMesh

In [16]:
#| hide

%load_ext jaxtyping
%jaxtyping.typechecker beartype.beartype

# enables type checking. does not work for some cells (vmapping and loading/saving). For those, %unload_ext jaxtyping

## `elastic` - Discrete elastic energies

This module defines elastic energies for thin shells, bilayers, and membranes for physics-based simulations. The energies are expressed directly using "discrete" quantities (like dihedral angles) which makes their computation (and optimization) on triangular meshes efficient.

For references, please see the following:
- ["Computing discrete shape operators on general meshes", E. Grinspun et al., 2006](cims.nyu.edu/gcl/papers/grinspun2006cds.pdf)
- "A discrete geometric view on shear-deformable shell models", C. Weischedel, 2012 (PhD thesis)
- ["Growth patterns for shape-shifting elastic bilayers", W. van Rees et al., 2017](http://www.pnas.org/cgi/doi/10.1073/pnas.1709025114)


### In-plane elastic energies

Thin elastic sheets can undergo very large deformation. Geometrically, the reference or rest shape of the sheet can be described by a metric tensor $g_0$.
The in-plane strain of a surface can be measured by the Cauchy-Green tensor $C_f = g_0^{-1} \cdot g_0$, computed from $g_0$ and $G$ the metric describing the actual configuration of the mesh.
A classic choice is the _neo-Hookean_ energy:
$$E_{\mathrm{NH}} = \sum_f A_f \left[ \frac{K}{2}\left(\frac{\mathrm{tr}(C_f)}{\sqrt{\det C_f}} - 2\right)
+\frac{B}{2}\left(\sqrt{\det C_f} - 1\right)^2
\right], \qquad C_f = g_0^{-1} g$$
$K$ and $B$ are the bulk and shear moduli. For $B=0$, only shear is penalized (the "Least Squares Conformal Mapping" (LSCM) energy). The metric can be easily evaluated for each triangle $f$ (see below). The total energy sums over all faces, weighted by area $A_f$. 
Another example is the _St.-Venant-Kirchhoff modell_:
$$E_{\mathrm{SK}} = \sum_f A_f \left[\alpha \, (\mathrm{tr}(C_f - \mathbb{I}))^2 + 2\beta \, \mathrm{tr}((C_f-\mathbb{I})^2) \right]$$
where $\alpha,\beta$ are the Lame coefficients.

#### Evaluating the metric for a triangle

The metric for a triangle $a, b, c$ can be computed from two side vectors $u=b-a, v=c-a$ as 
$$g_f = \begin{pmatrix}u^2 & u\cdot v \\ u\cdot v & v^2 \end{pmatrix} $$

**Important**: The reference metric, a $2\times 2$ matrix per triangle, also needs to be expressed in the "face basis" $u, v$.

In [ ]:
# --- Mesh energies ---

def get_metric(vertices: Float[jax.Array, "n_faces dim"],
               hemesh: HeMesh) -> Float[jax.Array, "n_faces 2 2"]:
    """Metric tensor (first fundamental form) per triangle."""
    a, b, c = vertices[hemesh.faces.T]
    J = jnp.stack([b - a, c - a], axis=1)
    return jnp.einsum("vik,vjk->vij", J, J)

def get_neo_hookean_energy_density(C: Float[jax.Array, "2 2"], mod_bulk: float, mod_shear: float
                                   ) -> Float[jax.Array, ""]:
    """Compute the neo-Hookean energy density from Cauchy-Green deformation tensor C:
        E = mod_shear/2 * (trace(C) / J - 2) + mod_bulk/2 * (J - 1)^2
    where J = sqrt(det(C)) is the local area change.
    See https://en.wikipedia.org/wiki/Neo-Hookean_solid
    """
    J = jnp.sqrt(jnp.linalg.det(C))
    return mod_shear/2*(jnp.trace(C)/J - 2) + mod_bulk/2*(J-1)**2

def get_neo_hookean_energy(vertices: Float[jax.Array, "n_faces dim"],
                           args: tuple[HeMesh, Float[jax.Array, "n_faces 2 2"], float, float]
                           ) -> Float[jax.Array, ""]:
    """Total neo-Hookean energy. args = (hemesh, metric_ref, mod_bulk, mod_shear)"""
    hemesh, metric_ref, mod_bulk, mod_shear = args
    areas = geom.get_triangle_areas(vertices, hemesh)
    metric = get_metric(vertices, hemesh)
    C = jnp.einsum("fij,fjk->fik", jnp.linalg.inv(metric_ref), metric)
    return (areas * jax.vmap(lambda c: get_neo_hookean_energy_density(c, mod_bulk, mod_shear))(C)).sum()

# to do: ensure the vectorization is is compatible with "broadcasting" of the coefficients mod_bulk and
# mod_shear, so that they can be specified either as a single scalar or per-triangle (inhomogenous moduli).

### Bending energies

Bending energies penalize out-of-plane deformation. A simple model uses the _dihedral angles_ $\theta_{e}$ for each edge (the angles between the normal vectors of adjacent triangles):
$$E_{\mathrm{B}} = \sum_e \frac{\ell_e^2}{A_e} (\theta_e -\theta_e^0)^2  $$
Where the area $A_e$ is defined as $1/3$ of the areas of the adjacent triangles. $\theta_e^0$ is the target bending angle.

A second option is to use the 2nd fundamental form $b_f$ (the extrinsic curvature tensor) and a variant of the St-Venant-Kirchhoff energy:
$$E_{\mathrm{B,SK}} = \sum_f A_f \left[\alpha \, (\mathrm{tr}(g_0^{-1}(b_f-b_0))^2 + 2\beta \, \mathrm{tr}((g_0^{-1}(b_f-b_0))^2) \right]$$
Here $b_0$ is the reference curvature. If the sheet prefers to be flat, $b_0 = 0$.

#### Evaluating the 2nd fundamental form for a triangle

For a triangle with vertices $a, b, c$, we defined the edge vectors $u=b-a, v=c-a$. For an edge $e$ between two faces $f, f'$, the per-edge normal is 
$$n_e = \frac{n_f+n_{f'}}{|n_f+n_{f'}|} $$
We label the triangle edges by the opposite vertex (so that $n_a$ is the normal of the edge opposite to vertex $a$). One obtains a finite-difference approximation for $b$:
$$b = \begin{pmatrix}2u\cdot(n_b-n_a) & u\cdot(n_c-n_a) + v\cdot(n_b-n_a) \\ u\cdot(n_c-n_a) + v\cdot(n_b-n_a) & 2v\cdot(n_c-n_a) \end{pmatrix} $$



In [ ]:
# to do: implement the discrete bending energies, both using dihedral angles and using the 
# discrete 2nd fundamental form. Follow the approach

# add test cases to verify the code runs correctly.

### Membrane energies

Membranes (e.g. lipid bilayers) differ from elastic shells because they are fluid in-plane: material can redistribute within a membrane to relax in-plane stresses through viscosity. However, membranes can still resist out-of-plane bending deformation.

The _Helfrich energy_ is an elegant, geometric model of bending energy. It uses the mean and Gaussian curvatures $H, K$ of the surface $\mathcal{M}$. The energy reads:

$$E_H  =\int dA \left( \frac{\kappa_H}{2}(H-H_0)^2 + \kappa_G K \right) $$

If the surface is closed, the $\int K$-term is a topological invariant and can be dropped (and we will do so here). A nonzero _spontaneous curvature_ $H_0$ means that the membrane "prefers" to be curved.

In [32]:
def get_helfrich_energy(vertices, args):
    """Compute the discrete Helfrich energy of a triangulated surface. args = (hemesh, H0, kappa)"""
    hemesh, H0, kappa_H, kappa_K = args
    # the cell areas are needed to discretize the area integral
    cell_areas = geom.get_voronoi_areas_robust(vertices, hemesh)
    #H = geom.get_mean_curvature_laplace(vertices, hemesh)
    H = geom.get_mean_curvature_dihedral(vertices, hemesh, normalize=True)
    K = geom.get_gaussian_curvature(vertices, hemesh)

    return (kappa_H/2) * ((H - H0) **2 * cell_areas).sum() + kappa_K * (K * cell_areas).sum()

In [33]:
#| hide

# load a test mesh
sphere = TriMesh.read_obj("../test_meshes/sphere_fine.obj", dim=3) # sphere_fine sphere
sphere.vertices -= sphere.vertices.mean(axis=0)
sphere.vertices = (sphere.vertices.T / np.linalg.norm(sphere.vertices, axis=1)).T

sphere_hemesh = HeMesh.from_triangles(sphere.vertices.shape[0], sphere.faces)

  o Icosphere


In [31]:
# Let's test the Helfrich energy on a spherical mesh.
# The exact energy for a sphere is 2*pi, here smaller due to discretization error.
# The energy is also scale invariant.

args = (sphere_hemesh, 0, 1, 1)

get_helfrich_energy(sphere.vertices, args), get_helfrich_energy(2*sphere.vertices, args), 2*np.pi

(Array(28.28502111, dtype=float64),
 Array(28.28502111, dtype=float64),
 6.283185307179586)